# 06 · Del scraping al dataset limpio
### Cómo convertí páginas de fútbol en datos listos para modelar

Este notebook no reproduce el pipeline completo ni requiere acceso a la base de datos privada. Es una versión explicable de la parte menos visible del proyecto: obtener HTML inconsistente, corregirlo y convertirlo en tablas confiables para los modelos posteriores.

> **Idea central:** el modelo no empieza con una predicción; empieza con datos que representan correctamente quién jugó, cuándo ocurrió cada evento y cómo identificar a cada jugador.

## 1. El problema: una página web no es un dataset

Para modelar un partido necesitaba información a tres niveles: metadatos del encuentro, secuencia de eventos y participación de cada jugador. FBref ofrece esa información, pero en páginas HTML diseñadas para lectura humana: tablas con encabezados repetidos, formatos distintos según el partido y nombres que no siempre coinciden.

El objetivo de esta etapa fue transformar cada reporte en registros consistentes, trazables y utilizables por los notebooks de RAG, contexto, sustituciones y simulación Monte Carlo.

## 2. Vista general del pipeline

```text
FBref
  │
  ├── Calendarios ──> fecha, equipos, URL del reporte
  └── Reportes   ──> alineaciones, tiros, xG, cambios, tarjetas y estadísticas
                         │
                         ▼
Selenium ──> BeautifulSoup / pandas ──> limpieza y validaciones ──> MySQL
```

Separar el proceso en dos fases permitió primero descubrir qué partidos existían y después enriquecer solamente los reportes que aún no tenían desglose de jugadores. Así se evita volver a procesar partidos completos sin necesidad.

### Dos pasos, una extracción incremental

```python
# Versión simplificada de la lógica de producción
fixtures = scrape_fixture_page(league_url)
save_new_matches(fixtures)

for match in matches_without_player_breakdown():
    html = render_match_report(match.url)
    extract_and_store_match_data(html, match.id)
```

El navegador automatizado fue necesario porque algunas tablas y pestañas del reporte se cargan o activan con JavaScript. Una vez obtenido el HTML renderizado, BeautifulSoup y pandas se encargan de localizar y estructurar el contenido.

## 3. Extraer un partido sin asumir que el HTML es perfecto

La extracción buscaba cuatro piezas: alineaciones iniciales, eventos, tabla de tiros y estadísticas individuales. La tabla de alineaciones era el camino principal; si no estaba disponible o cambiaba de formato, el pipeline reconstruía los jugadores desde las tablas de estadísticas.

```python
try:
    home_players = extract_players(lineup_table_home)
    away_players = extract_players(lineup_table_away)
except ParseError:
    # Fallback: cada tabla de estadísticas conserva jugador y dorsal
    home_players = extract_players_from_stats(html, home_team)
    away_players = extract_players_from_stats(html, away_team)
```

Esta decisión privilegia cobertura: si una presentación cambia, el partido no desaparece silenciosamente del historial.

## 4. Limpiar tablas: de etiquetas visuales a columnas analíticas

Las tablas de tiros podían llegar con encabezados multinivel, filas que repetían el encabezado dentro del cuerpo y valores como texto. La limpieza conserva únicamente las columnas necesarias y convierte cada campo según su significado.

| Antes | Después | Motivo |
|---|---|---|
| `('Unnamed: 0_level_0', 'Minute')` | `Minute` | Encabezado legible |
| `('Unnamed: 3_level_0', 'xG')` | `xG` | Variable numérica utilizable |
| fila `Minute / Squad / xG` | eliminada | Es un encabezado repetido, no un tiro |
| `''` en tarjetas o faltas | `0` | No hubo evento registrado |
| alineación no recuperable | `None` + fallback | El dato es desconocido, no cero |

### Ejemplo de normalización

```python
shots = shots[["Minute", "Squad", "xG"]].copy()
shots = shots[shots["Minute"].notna() & (shots["Minute"] != "Minute")]
shots["xG"] = shots["xG"].astype(float)

# Los conteos ausentes significan que no se registró ese evento.
player_stats[["fouls", "yellow_cards", "red_cards"]] = (
    player_stats[["fouls", "yellow_cards", "red_cards"]].fillna(0)
)
```

La regla importante no era llenar todos los nulos indiscriminadamente: `0` significa ausencia de una acción; `None` significa que la fuente no permitió determinar el dato.

## 5. Cinco dificultades reales

| Dificultad | Riesgo para el modelo | Respuesta del pipeline |
|---|---|---|
| Alineaciones con formatos variables | Jugadores ausentes o asignados al equipo equivocado | Extracción principal + fallback desde estadísticas |
| Minutos `45+N` y `90+N` | Eventos desordenados en la línea temporal | Normalizar y ajustar la segunda mitad |
| Nulos, texto y encabezados repetidos | Tipos erróneos y filas falsas | Limpieza semántica por columna |
| Nombres distintos entre fuentes | No encontrar al jugador correcto | Normalización y fuzzy matching dentro del equipo |
| Dorsales cambiantes | Duplicar el historial de un jugador | Unificación conservadora de IDs |

### 5.1 Una línea temporal única para todo el partido

Un `45+2` no es solamente texto: representa el minuto 47. Además, el tiempo añadido de la primera mitad desplaza los eventos posteriores cuando el modelo trabaja con una secuencia continua de minutos.

| Evento original | Minuto normalizado | Minuto en la línea continua si hubo `45+2` |
|---|---:|---:|
| Gol `45+2` | 47 | 47 |
| Cambio `60` | 60 | 62 |
| Tiro `90+4` | 94 | 96 |

```python
def parse_minute(value):
    base, *extra = str(value).split("+")
    return int(base) + (int(extra[0]) if extra else 0)

def adjust_second_half(minute, first_half_stoppage):
    return minute + first_half_stoppage if minute > 45 else minute
```

### 5.2 Matching de nombres: conectar la fuente con el ID interno

Las alineaciones ingresadas desde una fuente externa pueden escribir un nombre distinto al registrado en la base. El matching se limita al plantel del equipo: primero normaliza mayúsculas, acentos y espacios; después busca la mejor coincidencia por similitud. Un ID ya asignado se retira de los candidatos para no asignar dos veces al mismo jugador.

```python
normalized = normalize_name(raw_name)
candidate = fuzzy_match(normalized, remaining_team_names, threshold=85)

if candidate:
    remaining_team_names.remove(candidate)
    return candidate.player_id
return raw_name  # queda visible para revisión, no se inventa un ID
```

El umbral se relaja gradualmente solo para los nombres que quedaron pendientes; así los matches obvios se resuelven primero.

## 6. Caso destacado: el mismo jugador con dos IDs

El ID de un jugador se construye con nombre, dorsal e iniciales del equipo. Eso es útil para identificar una fila durante el scraping, pero crea un problema si el dorsal cambia entre partidos.

| Partido | ID extraído | Lectura humana |
|---:|---|---|
| 120 | `Álvaro_García_18_RV` | Álvaro García, dorsal 18 |
| 184 | `Álvaro_García_7_RV` | Álvaro García, dorsal 7 |

Sin limpieza, el modelo interpretaría dos jugadores distintos y dividiría sus minutos, estadísticas y coeficientes.

### La heurística de unificación

1. Agrupar IDs por equipo y nombre, ignorando el dorsal.
2. Revisar si dos candidatos aparecen juntos en la misma alineación. Si aparecen juntos, se conservan: podrían ser dos personas diferentes.
3. Si nunca aparecen juntos, tratarlos como el mismo jugador con otro dorsal.
4. Conservar el ID usado en el partido más reciente como ID canónico.
5. Reescribir las referencias históricas y eliminar el ID obsoleto.

```python
candidates = group_by_team_and_name(player_ids)

for ids in candidates:
    if len(ids) < 2 or appear_together_in_a_match(ids):
        continue

    canonical_id = most_recent_id(ids)
    obsolete_ids = set(ids) - {canonical_id}
    rewrite_references(obsolete_ids, canonical_id)
```

### Antes y después: una identidad canónica

| Tabla | Antes | Después |
|---|---|---|
| `players` | `Álvaro_García_18_RV`, `Álvaro_García_7_RV` | `Álvaro_García_7_RV` |
| `match_detailed` | listas con ambos IDs según la fecha | todas las listas usan el ID canónico |
| `match_player_breakdown` | estadísticas repartidas entre dos IDs | historial bajo una sola identidad |

```sql
-- La versión de producción actualiza primero las referencias.
UPDATE match_player_breakdown
SET player_id = :canonical_id
WHERE player_id IN (:obsolete_ids);

DELETE FROM players
WHERE id IN (:obsolete_ids);
```

Es una heurística deliberadamente conservadora, no una identidad perfecta. Dos jugadores homónimos que nunca coinciden podrían fusionarse por error; las transferencias entre equipos tampoco se resuelven automáticamente; y usar el mayor `match_id` presupone que los IDs siguen el orden temporal. Esos límites deben revisarse si se amplía la cobertura del proyecto.

## 7. Dónde termina cada dato

| Tabla | Granularidad | Contenido principal | Uso posterior |
|---|---|---|---|
| `match_general` | Un registro por partido | fecha, equipos, liga y URL | calendario e historial de encuentros |
| `match_detailed` | Un registro por segmento | jugadores activos, xG, marcador, minutos y contexto | Raw/Contextual RAG y simulación |
| `match_player_breakdown` | Jugador-partido | minutos, cambios, faltas y tarjetas | fatiga, ritmo y comportamiento individual |
| `players` | Jugador canónico | identidad y métricas agregadas | coeficientes y consultas de plantilla |

La clave no es guardar HTML transformado: es guardar una versión estructurada que conecte consistentemente partido, segmento y jugador.

## 8. Validaciones y cierre

Antes de que los datos pasen al modelado, el pipeline verifica que las fechas estén en el rango esperado, que los minutos sean válidos, que un partido no se procese dos veces y que los jugadores referenciados existan. La regla de duplicados añade una validación específica: no unificar IDs que aparecieron juntos en un mismo partido.

**Tres ideas que me llevé de esta etapa:**

1. Un scraper útil necesita alternativas cuando la fuente cambia, no solo un selector CSS que funcione hoy.
2. Limpiar datos es decidir qué significa cada ausencia, formato o inconsistencia.
3. La identidad histórica de los jugadores es tan importante como las métricas: sin ella, el modelo aprende señales fragmentadas.

Con esta capa de extracción y limpieza, los notebooks siguientes pueden concentrarse en analizar y modelar el fútbol, no en corregir el HTML de origen.